# Option Pricing Showcase
Demonstrate various option pricing capabilities

In [ ]:
from utils.pricing_utils import price_vanilla_options, price_barrier_options, calculate_option_greeks, generate_payoff_data
from utils.market_data_utils import fetch_spot_price
from utils.common import timer
import pandas as pd
import matplotlib.pyplot as plt

## Setup Pricing Parameters

In [ ]:
# Real market data
ticker = 'AAPL'
market_data = fetch_spot_price(ticker)

# Pricing parameters
spot = market_data['spot']
strike = round(spot)  # ATM
time_to_expiry = 30/365  # 30 days
volatility = 0.25  # 25%
risk_free_rate = 0.045  # 4.5%

print(f"Pricing Parameters for {ticker}:")
print(f"  Spot: ${spot:.2f}")
print(f"  Strike: ${strike:.2f}")
print(f"  Time: {time_to_expiry*365:.0f} days")
print(f"  Volatility: {volatility:.1%}")
print(f"  Rate: {risk_free_rate:.2%}")

## European vs American Options

In [ ]:
@timer
def price_all_vanillas():
    return price_vanilla_options(spot, strike, time_to_expiry, volatility, risk_free_rate)

prices = price_all_vanillas()

print("Option Prices:")
print("=" * 40)
print(f"European Call: ${prices['european_call']:.2f}")
print(f"American Call: ${prices['american_call']:.2f}")
print(f"Early Exercise Premium: ${prices['early_exercise_premium_call']:.2f}")
print()
print(f"European Put: ${prices['european_put']:.2f}")
print(f"American Put: ${prices['american_put']:.2f}")
print(f"Early Exercise Premium: ${prices['early_exercise_premium_put']:.2f}")

## Barrier Options

In [ ]:
# Down-and-out barrier
barrier_down = spot * 0.9  # 10% below spot

@timer
def price_barriers():
    return price_barrier_options(spot, strike, barrier_down, time_to_expiry, volatility, risk_free_rate)

barrier_prices = price_barriers()

print(f"Barrier Option Pricing (Barrier at ${barrier_down:.2f}):")
print("=" * 40)
if 'down_out_call' in barrier_prices:
    print(f"Down-and-Out Call: ${barrier_prices['down_out_call']:.2f}")
    print(f"Vanilla Call: ${barrier_prices['vanilla_call']:.2f}")
    print(f"Discount: {barrier_prices['down_out_call_discount']:.1f}%")

## Option Greeks

In [ ]:
greeks_call = calculate_option_greeks(spot, strike, time_to_expiry, volatility, risk_free_rate, is_call=True)
greeks_put = calculate_option_greeks(spot, strike, time_to_expiry, volatility, risk_free_rate, is_call=False)

# Display as table
greeks_df = pd.DataFrame({
    'Greek': ['Delta', 'Gamma', 'Theta', 'Vega', 'Rho'],
    'Call': [f"{greeks_call['delta']:.4f}",
             f"{greeks_call['gamma']:.4f}",
             f"{greeks_call['theta']:.4f}",
             f"{greeks_call['vega']:.4f}",
             f"{greeks_call['rho']:.4f}"],
    'Put': [f"{greeks_put['delta']:.4f}",
            f"{greeks_put['gamma']:.4f}",
            f"{greeks_put['theta']:.4f}",
            f"{greeks_put['vega']:.4f}",
            f"{greeks_put['rho']:.4f}"]
})

print(greeks_df.to_string(index=False))

## Payoff Diagram

In [ ]:
# Generate payoff data
call_premium = prices['european_call']
payoff_data = generate_payoff_data('call', strike, call_premium)

# Simple plot
plt.figure(figsize=(10, 6))
plt.plot(payoff_data['spot_prices'], payoff_data['payoffs'], 'b-', linewidth=2)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=strike, color='red', linestyle='--', alpha=0.5, label=f'Strike: ${strike}')
plt.axvline(x=payoff_data['breakeven'], color='green', linestyle='--', alpha=0.5, label=f'Breakeven: ${payoff_data["breakeven"]:.2f}')

plt.xlabel('Spot Price ($)')
plt.ylabel('P&L ($)')
plt.title(f'Call Option Payoff (Strike: ${strike}, Premium: ${call_premium:.2f})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Max Loss: ${payoff_data['max_loss']:.2f}")
print(f"Breakeven: ${payoff_data['breakeven']:.2f}")

## Multi-Leg Strategy: Iron Condor

In [ ]:
# Iron Condor strikes
strikes = {
    'put_sell': spot * 0.95,
    'put_buy': spot * 0.90,
    'call_sell': spot * 1.05,
    'call_buy': spot * 1.10
}

# Price each leg
iron_condor_legs = {}
for leg, strike_price in strikes.items():
    is_call = 'call' in leg
    from derivatives_gpt_core.features.vanilla.pricing import price_european_option
    price = price_european_option(spot, strike_price, time_to_expiry, risk_free_rate, volatility, is_call)
    iron_condor_legs[leg] = {'strike': strike_price, 'price': price}

# Calculate net premium
net_premium = (iron_condor_legs['put_sell']['price'] - iron_condor_legs['put_buy']['price'] +
               iron_condor_legs['call_sell']['price'] - iron_condor_legs['call_buy']['price'])

print("Iron Condor Pricing:")
print("=" * 40)
for leg, data in iron_condor_legs.items():
    action = 'Sell' if 'sell' in leg else 'Buy'
    print(f"{action} {leg.split('_')[0].title()}: Strike ${data['strike']:.2f}, Premium ${data['price']:.2f}")

print(f"\nNet Premium Received: ${net_premium:.2f}")
print(f"Max Profit: ${net_premium:.2f}")
print(f"Max Loss: ${(strikes['call_buy'] - strikes['call_sell'] - net_premium):.2f}")